# Libraries Required

In [ ]:
import tkinter as tk
import threading
import sounddevice as sd
import numpy as np
import os
import wave
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
from scipy.io.wavfile import write
import nemo.collections.asr as nemo_asr
from tensorflow.keras.models import load_model
import librosa

C:\Users\Harsh\anaconda3\envs\nemo_env\lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
C:\Users\Harsh\anaconda3\envs\nemo_env\lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
[NeMo W 2025-04-16 09:31:23 transformer_bpe_models:59] Could not import NeMo NLP collection which is required for speech translation model.


[NeMo I 2025-04-16 09:31:28 mixins:170] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2025-04-16 09:31:30 modelPT:161] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: null
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 40
    min_duration: 0.1
    is_tarred: true
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    shard_manifests: true
    use_lhotse: true
    use_bucketing: true
    num_buckets: 30
    bucket_duration_bins: null
    batch_duration: 600
    defer_setup: true
    
[NeMo W 2025-04-16 09:31:30 modelPT:168] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
  

[NeMo I 2025-04-16 09:31:30 features:289] PADDING: 0


[NeMo W 2025-04-16 09:31:31 nemo_logging:349] C:\Users\Harsh\anaconda3\envs\nemo_env\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
      warnings.warn(
    


[NeMo I 2025-04-16 09:31:31 rnnt_models:211] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2025-04-16 09:31:33 save_restore_connector:249] Model EncDecHybridRNNTCTCBPEModel was successfully restored from C:\Users\Harsh\.cache\huggingface\hub\models--nvidia--parakeet-tdt_ctc-110m\snapshots\431a349f3051ab85c22b9b7a2741b5fe77065665\parakeet-tdt_ctc-110m.nemo.


[NeMo W 2025-04-16 09:31:33 nemo_logging:349] C:\Users\Harsh\anaconda3\envs\nemo_env\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
      warnings.warn(
    


# Important Functions

In [1]:
# Path to save recorded audio
AUDIO_DIR = "C:/lenovo-ideapad/CUDA_PJT/Recorded_audio/"
os.makedirs(AUDIO_DIR, exist_ok=True)
AUDIO_PATH_1 = os.path.join(AUDIO_DIR, "recorded_audio.wav")
OUTPUT_FILE = os.path.join(AUDIO_DIR, "transcription.txt")
# Both Parakeet and FLAN-T5 are using internet
asr_model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(model_name="nvidia/parakeet-tdt_ctc-110m")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
# This model is created using the SER.ipynb
ser_model = load_model("trained_model.h5")
emotion_labels = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprise']

# Audio settings
SAMPLERATE = 16000  # 16kHz
CHANNELS = 1  # Mono
recording = False
audio_data = []

def callback(indata, frames, time, status):
    """Callback function to store recorded audio chunks."""
    if status:
        print(status)
    if recording:
        audio_data.append(indata.copy())

def start_recording():
    """Starts recording in a separate thread."""
    global recording, audio_data
    if not recording:
        recording = True
        audio_data = []
        threading.Thread(target=record_audio, daemon=True).start()
        status_label.config(text="Recording...", fg="red")

def record_audio():
    """Records audio using sounddevice."""
    with sd.InputStream(samplerate=SAMPLERATE, channels=CHANNELS, dtype=np.int16, callback=callback):
        while recording:
            sd.sleep(100)

def stop_recording():
    """Stops recording and saves the file."""
    global recording
    if recording:
        recording = False
        # Convert recorded data to NumPy array
        audio_np = np.concatenate(audio_data, axis=0)
        # Save the recorded audio
        write(AUDIO_PATH_1, SAMPLERATE, audio_np)
        status_label.config(text="Recording stopped. Ready.", fg="green")
        print(f"Recording saved to {AUDIO_PATH_1}")

        # Start transcription
        threading.Thread(target=transcribe_audio, daemon=True).start()

def detect_emotion(text):
    # Sends the prompt to FLAN-T5 (LLM) for output
    prompt = f"""What is the emotion of this sentence?
    \nGive the answer from among these labels: ['neutral','calm', 'happy', 'sad', 'angry', 'fearful','disgust','surprise']
    \n\n{text}"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=50)
    emotion = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return emotion

def detect_ser_emotion(file_path):
    audio, sr = librosa.load(file_path, sr=16000)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=1)
    mfccs = np.pad(mfccs, ((0, 0), (0, max(0, 162 - mfccs.shape[1]))), mode='constant')
    mfccs = mfccs[:, :162]
    mfccs = np.expand_dims(mfccs.T, axis=-1)
    mfccs = np.expand_dims(mfccs, axis=0)
    predictions = ser_model.predict(mfccs)
    predicted_class = np.argmax(predictions, axis=1)
    return emotion_labels[predicted_class[0]]

def transcribe_audio():
    try:
        status_label.config(text="Transcribing...", fg="blue")

        # Transcribe using Parakeet
        result = asr_model.transcribe([AUDIO_PATH_1])
        print("Raw result from model:", result)
        transcript = result[0][0].strip()

        # Save transcription
        with open(OUTPUT_FILE, "w") as f:
            f.write(transcript)

        # Emotion Detection: FLAN-T5
        asr_emotion = detect_emotion(transcript)

        # Emotion Detection: SER model
        ser_emotion = detect_ser_emotion(AUDIO_PATH_1)

        # Save both emotions
        with open(os.path.join(AUDIO_DIR, "emotion_asr.txt"), "w") as f:
            f.write(asr_emotion)
        with open(os.path.join(AUDIO_DIR, "emotion_ser.txt"), "w") as f:
            f.write(ser_emotion)

        # Update GUI
        status_label.config(text="Transcription & Emotion Detection Complete!", fg="green")
        transcript_label.config(
            text=f""""Transcription: {transcript}\n\nEmotion (ASR): {asr_emotion}\nEmotion (SER): {ser_emotion}
            \n The user seems {asr_emotion}, but sounds {ser_emotion}."""
        )

        print(f"Detected Emotion (ASR): {asr_emotion}")
        print(f"Detected Emotion (SER): {ser_emotion}")

    except Exception as e:
        status_label.config(text="Error in transcription or emotion detection!", fg="red")
        print(f"Error: {e}")


NameError: name 'os' is not defined

# Tkinter App for Emotion Recognition

In [ ]:
# Create the Tkinter app
root = tk.Tk()
root.title("Audio Recorder & Transcriber")
root.geometry("600x400")

# Frame for buttons
button_frame = tk.Frame(root)
button_frame.pack(pady=20)

# Start button (Left)
start_button = tk.Button(button_frame, text="Start Recording", font=("Arial", 14), padx=20, pady=10, command=start_recording)
start_button.pack(side="left", padx=10)

# Stop button (Right)
stop_button = tk.Button(button_frame, text="Stop Recording", font=("Arial", 14), padx=20, pady=10, command=stop_recording)
stop_button.pack(side="right", padx=10)

# Status label
status_label = tk.Label(root, text="Ready.", fg="green", font=("Arial", 12))
status_label.pack(pady=10)

# Transcription output
transcript_label = tk.Label(root, text="", wraplength=500, justify="left", font=("Arial", 12))
transcript_label.pack(pady=10)

# Keep the Tkinter window open
root.mainloop()